1. Importing Libraries

In [1]:
from langchain.document_loaders import PyPDFLoader, TextLoader, CSVLoader, JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA

2. Installing Formats

In [2]:
!pip install unstructured
!pip install "unstructured[pdf]"
!pip install "unstructured[local-inference]"


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install jq


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


3. Load CSV Data

In [4]:
csv_loader = CSVLoader("D:/OneDrive - Coforge Limited/Desktop/AI Training/Medical/format_dataset.csv")

In [5]:
csv_document = csv_loader.load()

4. Load JSON Data

In [6]:
jq_schema = ".[] | {instruction: .instruction, input: .input, output: .output}"
 
medical_json_loader = JSONLoader(file_path="./Medical/chatdoctor5k.json",
                            jq_schema=jq_schema, text_content = False)

In [7]:
medical_json_docs = medical_json_loader.load()

5. Concatinating the files

In [8]:
documents = csv_document + medical_json_docs

In [9]:
print(documents[0])

page_content='idx: 0
disease: Panic disorder
Symptom: ['Anxiety and nervousness', 'Depression', 'Shortness of breath', 'Depressive or psychotic symptoms', 'Sharp chest pain', 'Dizziness', 'Insomnia', 'Abnormal involuntary movements', 'Chest tightness', 'Palpitations', 'Irregular heartbeat', 'Breathing fast']
reason: Panic disorder is an anxiety disorder characterized by recurring severe panic attacks. It may also include significant behavioral changes lasting at least a month and of ongoing worry about the implications or concern about having other attacks. The latter are called anticipatory attacks (DSM-IVR). Panic disorder is not the same as agoraphobia (fear of public places), although many afflicted with panic disorder also suffer from agoraphobia. Panic attacks cannot be predicted, therefore an individual may become stressed, anxious or worried wondering when the next panic attack will occur. Panic disorder may be differentiated as a medical condition, or chemical imbalance. The D

In [10]:
print(f"Number of documents loaded from directory : {len(documents)}")

Number of documents loaded from directory : 6248


6. Applying Recursive TextSplitter

In [11]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ""])

In [12]:
recursive_tokens = recursive_splitter.split_documents(documents)

In [13]:
# size of the tokens
print(f"Number of tokens created from document: {len(recursive_tokens)}")

Number of tokens created from document: 17976


7. Embeddings Using OpenAI

In [29]:
!pip install langchain-openai==0.3.27


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


8. Applying LLM

In [ ]:
# setup llm
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [37]:
from langchain_openai import OpenAIEmbeddings

##### APPLY EMBEDINGS

In [38]:
openai_embeddings = OpenAIEmbeddings(model = "text-embedding-ada-002",
                                     openai_api_key = os.environ["OPENAI_API_KEY"])

In [43]:
faiss_store = FAISS.from_documents(documents = recursive_tokens,
                                   embedding = openai_embeddings)

9. Buffer Memory

In [44]:
from langchain.memory import ConversationBufferMemory

In [45]:
memory = ConversationBufferMemory(memory_key="chat_history",
                                  return_messages=True, output_key='answer')

C:\Users\Shivam.2.Goyal\AppData\Local\Temp\ipykernel_24960\1504241007.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history",


10. Search Technique

In [46]:
# similarity : Rank chunks purely by cosine distnace between the query and
# the chunk embeddings. This is the default
# search type. The top k most similar chunks are returned closest to the
# query vector.
 
# retriever = faiss_store.as_retriever(search_type="similarity", search_kwargs={"k":3})
 
# max marginal relevance (mmr): Rank chunks by a combination of their
# similarity to the query and their diversity relative to the already
# selected chunks. This does a two step process, it first feteches a larger
# candidate pool then iteratively selects the most relevant and diverse
# chunks until it has selected k chunks. This search type is designed to
# improve the diversity of the results while maintaining relevance.
retriever = faiss_store.as_retriever(search_type="mmr", search_kwargs={"k":3})

11. create prompt template

In [47]:
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template="""You are a medical AI assistant trained on a curated medical knowledge dataset.
Your role is to provide reliable, understandable, and evidence-based health information. You are not a replacement for a qualified doctor or medical professional.


INSTRUCTIONS:
1. Understand the user's symptoms and medical context carefully.
2. Provide information based on the medical knowledge learned by the model.
3. Explain medical terms in simple language.
4. Do not claim certainty when the information is insufficient.
5. Do not make a definitive diagnosis solely from symptoms.
6. Provide possible causes or conditions as possibilities, not confirmed diagnoses.
7. If appropriate, suggest what type of healthcare professional the user should consult.
8. Clearly identify emergency symptoms that require immediate medical attention.
9. Do not recommend prescription medicines or dosage changes without appropriate medical supervision.
10. If important information is missing, ask relevant follow-up questions.
11. Never fabricate medical facts, test results, medical history, or references.
RESPONSE FORMAT:
1. **Understanding the concern**
  - Briefly summarize the user's issue.
2. **Possible explanations**
  - List the most relevant possibilities.
  - Explain each in simple language.
3. **What you can do now**
  - Provide safe, general self-care or next steps where appropriate.
4. **When to see a doctor**
  - Explain when professional medical evaluation is recommended.
5. **Emergency warning**
  - Clearly state if symptoms could indicate an emergency and what action to take.
6. **Follow-up questions**
  - Ask only the questions needed to better understand the situation.
DISCLAIMER:
This information is for educational purposes and does not replace diagnosis or treatment from a qualified healthcare professional.
 
    Context:
    {context}
 
    Chat History:
    {chat_history}
 
    Question: {question}
 
    Answer:""")
 

12. RAG Creation

In [49]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain

In [50]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4.1-nano",
                 temperature=0.5,
                 top_p=0.75)

In [51]:
QA_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,      # to return the source documents along with the answer
    verbose=False,
    combine_docs_chain_kwargs={"prompt": prompt},
    output_key='answer'
)
print("Welcome to the Disease Analysis CHATBOT!")

Welcome to the Disease Analysis CHATBOT!


In [52]:
def ask_question(question: str) -> dict:
    result = QA_chain.invoke({"question": question})
   
    print("=" * 70)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print("-" * 70)
    print("Sources:")
    for i, doc in enumerate(result["source_documents"], start=1):
        meta   = doc.metadata
        source = meta.get("source", "Unknown").split("\\")[-1]  # filename only
        page   = meta.get("page", "?")
        print(f"  [{i}] {source}  |  Page {page}")
        print(f"       {doc.page_content[:150].strip()}...")
    print("=" * 70)
 

In [57]:
ask_question("What was the disease i am having I think I fractured my hand while playing basketbal ?")

Q: What was the disease i am having I think I fractured my hand while playing basketbal ?
A: Based on your description of swelling and inability to move your hand properly after a basketball game, it is possible that you have an injury to your hand, such as a fracture or sprain. I recommend seeking medical attention promptly for imaging studies to confirm the diagnosis and determine the appropriate treatment.
----------------------------------------------------------------------
Sources:
  [1] chatdoctor5k.json  |  Page ?
       . What could be wrong with me?", "output": "Based on your symptoms, it is possible that you have an injury to your hand, which could include a fractur...
  [2] D:/OneDrive - Coforge Limited/Desktop/AI Training/Medical/format_dataset.csv  |  Page ?
       idx: 27
disease: Injury to the hand...
  [3] chatdoctor5k.json  |  Page ?
       {"instruction": "If you are a doctor, please answer the medical questions based on the patient's description.", "input": "Doctor,